In [ ]:
"""
03_quantisation_analysis.py
────────────────────────────
Quantisation sweep on SAE bottleneck activations, with perplexity via
activation patching, MSE, CKA, and SDS metrics.

Targets: Assignment Section 2 (Quantisation Analysis).

Paste ActivationNormalizer and the distilgpt2 loading code from
01_pipeline_setup.py into this notebook before running.

Section map
  1.  Config
  2.  Load model, SAE, normalizer
  3.  Held-out data collection
  4.  Smoke tests               ← run and verify before proceeding
  5.  Quantisation functions
  6.  Hook infrastructure
  7.  Metrics (MSE, CKA, SDS)
  8.  Experiment loop
  9.  UMAP visualisation
  10. Results table
"""

# ── 1. Config ─────────────────────────────────────────────────────────────────
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional, Callable

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
LAYER_IDX = 3
D_MODEL   = 768
M         = 512
K         = M // 10

# Update these paths after attaching the relevant Kaggle dataset outputs
NOTEBOOK01_DIR = Path("/kaggle/input/notebooks/codemtc/pipeline-setup/activations")    # normalizer.pt
NOTEBOOK02_DIR = Path("/kaggle/input/notebooks/codemtc/sae-training-m512") # sae checkpoint
OUT_DIR        = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

HELD_OUT_SEQS  = 2_000    # sequences for perplexity evaluation (~256k tokens)
EVAL_BATCH     = 32       # sequences per forward pass during evaluation
METRIC_TOKENS  = 50_000   # tokens for MSE / CKA / SDS metric computation
UMAP_TOKENS    = 10_000   # tokens for UMAP visualisation
SDS_K_VALUES   = [32, 64, 128]
BITWIDTHS      = [8, 4, 2]
QUANT_TYPES    = ["per_tensor", "per_feature"]

### Classes from previous notebooks

In [ ]:
class ActivationNormalizer:
    """
    Per-dimension (channel-wise) z-score normalizer.

    Fit on the debug pass (~200k tokens); save and reuse for the full extraction
    and at eval time. Consistent normalization is critical: the SAE learns a
    dictionary in the normalized space, so test-time patching must use the same
    mean/std.

    Why per-dimension?  distilgpt2 layer-3 output dimensions have very different
    scales — some are near-zero throughout the dataset, others span ±10+.
    Global (scalar) normalization leaves that structure intact, making the SAE
    learn an uneven dictionary. Per-dimension normalization is what Anthropic and
    Nanda's public SAE implementations both use.
    """

    def __init__(self):
        self.mean: Optional[torch.Tensor] = None
        self.std:  Optional[torch.Tensor] = None

    def fit(self, acts: torch.Tensor, eps: float = 1e-6):
        """acts: (N, D_MODEL), any dtype."""
        a = acts.float()
        self.mean = a.mean(dim=0)                    # (D_MODEL,)
        self.std  = a.std(dim=0).clamp(min=eps)      # (D_MODEL,)
        print(
            f"  normalizer: mean.norm={self.mean.norm():.3f}  "
            f"std.mean={self.std.mean():.4f}  "
            f"std.min={self.std.min():.6f}"
        )

    def __call__(self, acts: torch.Tensor) -> torch.Tensor:
        """Return normalized activations in the same dtype as input."""
        dtype = acts.dtype
        a = acts.float()
        out = (a - self.mean.to(a.device)) / self.std.to(a.device)
        return out.to(dtype)

    def inverse(self, acts: torch.Tensor) -> torch.Tensor:
        dtype = acts.dtype
        a = acts.float()
        out = a * self.std.to(a.device) + self.mean.to(a.device)
        return out.to(dtype)

    def save(self, path: Path):
        torch.save({"mean": self.mean, "std": self.std}, path)
        print(f"  normalizer saved → {path}")

    @classmethod
    def load(cls, path: Path) -> "ActivationNormalizer":
        n = cls()
        ckpt = torch.load(path, map_location="cpu")
        n.mean, n.std = ckpt["mean"], ckpt["std"]
        return n

In [ ]:
class SparseAutoencoder(nn.Module):
    """
    Top-k Sparse Autoencoder.

    Architecture (tied-bias formulation, standard in mech interp):
        z    = topk( ReLU( W_enc @ (x - b_dec) + b_enc ) )
        x̂   = W_dec @ z + b_dec
        loss = MSE(x, x̂)

    The pre-encoder bias (b_dec) is subtracted before encoding and added
    back after decoding. This centers the input around the decoder's
    natural origin so the encoder learns directions, not offsets.

    Decoder columns (dictionary atoms) are kept at unit norm after every
    gradient step. Without this constraint, the trivially optimal solution
    is to make decoder atoms very large and encoder weights very small,
    which achieves low MSE without learning anything meaningful.

    Top-k vs L1
    ───────────
    The assignment specifies top-k sparsity rather than the L1 penalty used
    in Anthropic's original paper. Top-k has two advantages here: (1) it
    guarantees exactly K active features per token rather than varying
    sparsity, making L0 a constant diagnostic rather than a tunable one;
    (2) it removes the L1 coefficient as a hyperparameter to tune.
    """

    def __init__(self, d_in: int = D_MODEL, m: int = M, k: int = K):
        super().__init__()
        self.d_in = d_in
        self.m    = m
        self.k    = k

        # Encoder weights and bias
        self.W_enc = nn.Parameter(torch.empty(d_in, m))
        self.b_enc = nn.Parameter(torch.zeros(m))

        # Decoder weights (columns = dictionary atoms) and shared bias
        self.W_dec = nn.Parameter(torch.empty(d_in, m))
        self.b_dec = nn.Parameter(torch.zeros(d_in))

        self._init_weights()

    def _init_weights(self):
        nn.init.kaiming_uniform_(self.W_enc, nonlinearity="relu")
        # Initialize decoder as encoder transpose then normalize.
        # This gives a reasonable starting point where encoder and decoder
        # are approximately inverses of each other.
        with torch.no_grad():
            self.W_dec.data = self.W_enc.data.T.clone()
            self._normalize_decoder()

    @torch.no_grad()
    def _normalize_decoder(self):
        """Normalize each column of W_dec to unit norm in-place."""
        # W_dec: (m, d_in) — normalize along d_in dimension (dim=1)
        norms = self.W_dec.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self.W_dec.data /= norms

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, d_in) → z: (batch, m)
        z has exactly K non-zero entries per row.
        """
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc  # (batch, m)
        acts = F.relu(pre)

        # Hard top-k: scatter the k largest values, zero the rest.
        # sorted=False is faster and order doesn't matter here.
        topk_vals, topk_idx = acts.topk(self.k, dim=-1, sorted=False)
        z = torch.zeros_like(acts)
        z.scatter_(-1, topk_idx, topk_vals)
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """z: (batch, m) → x̂: (batch, d_in)"""
        return z @ self.W_dec + self.b_dec

    def forward(self, x: torch.Tensor):
        z    = self.encode(x)
        x_hat = self.decode(z)
        loss  = F.mse_loss(x_hat, x)
        return loss, z, x_hat

In [ ]:
# ── 2. Load model, SAE, normalizer ────────────────────────────────────────────
# (paste ActivationNormalizer and SparseAutoencoder class definitions
#  from notebooks 01 and 02 above this block)

def load_frozen_model(device=DEVICE):
    tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        "distilgpt2", dtype=torch.float32
    ).to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model, tokenizer


def load_sae(path: Path, device=DEVICE):
    ckpt = torch.load(path, map_location=device)
    cfg  = ckpt["config"]
    sae  = SparseAutoencoder(d_in=cfg["d_in"], m=cfg["m"], k=cfg["k"]).to(device)
    sae.load_state_dict(ckpt["model_state"])
    sae.eval()
    for p in sae.parameters():
        p.requires_grad_(False)
    print(f"  SAE loaded: m={cfg['m']} k={cfg['k']} from step {ckpt['step']:,}")
    return sae

In [ ]:
# ── 3. Held-out data ──────────────────────────────────────────────────────────
SEQ_LEN = 128

def _token_windows(tokenizer, max_seqs: int, skip_tokens: int = 12_000_000):
    """
    Yield non-overlapping 128-token windows, skipping the first `skip_tokens`
    tokens so the held-out set does not overlap with the 10M training tokens.
    """
    ds = load_dataset(
        "Skylion007/openwebtext", split="train", streaming=True
    )
    carry   = torch.empty(0, dtype=torch.long)
    skipped = 0
    yielded = 0

    for ex in ds:
        if yielded >= max_seqs:
            break
        ids = tokenizer.encode(ex["text"], add_special_tokens=False,
                               return_tensors="pt").squeeze(0)
        ids = torch.cat([carry, ids, torch.tensor([tokenizer.eos_token_id])])

        n = len(ids) // SEQ_LEN
        for i in range(n):
            window = ids[i * SEQ_LEN : (i + 1) * SEQ_LEN]
            if skipped < skip_tokens:
                skipped += SEQ_LEN
                continue
            if yielded >= max_seqs:
                break
            yield window
            yielded += 1
        carry = ids[n * SEQ_LEN :]


@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids

In [ ]:
# ── 4. Smoke tests ────────────────────────────────────────────────────────────
def run_smoke_tests(model, tokenizer, sae, normalizer, held_out_ids):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    # ── Test 1: distilgpt2 baseline perplexity ────────────────────────────────
    # Without any patching, perplexity should be reasonable (~20-60 for OWT)
    print("\n[1] Baseline perplexity (no patching)")
    ppl = _perplexity_clean(model, held_out_ids[:200])
    print(f"    perplexity = {ppl:.2f}")
    assert 15 < ppl < 200, f"perplexity {ppl:.2f} is outside expected range"
    print("    ✓")

    # ── Test 2: identity patch (SAE encode→decode, no quantisation) ───────────
    # Perplexity should be slightly higher than baseline (SAE is not perfect),
    # but not dramatically so. A well-trained SAE typically adds 2-10% PPL.
    print("\n[2] Identity patch (SAE reconstruction, no quantisation)")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                  quantise_fn=None)
    overhead = 100 * (ppl_sae / ppl - 1)
    print(f"    perplexity = {ppl_sae:.2f}  ({overhead:+.1f}% vs baseline)")
    assert ppl_sae > ppl, "SAE-patched PPL should be >= clean PPL"
    assert ppl_sae < ppl * 5, "SAE reconstruction is unexpectedly bad"
    print("    ✓")

    # ── Test 3: 8-bit quantisation should be close to identity patch ──────────
    print("\n[3] 8-bit per-tensor quantisation")
    q8 = make_quantise_fn("per_tensor", bits=8)
    ppl_8bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q8)
    print(f"    perplexity = {ppl_8bit:.2f}")
    assert ppl_8bit < ppl * 10, "8-bit PPL is implausibly high — check quantisation"
    print("    ✓")

    # ── Test 4: 2-bit should be worse than 8-bit ─────────────────────────────
    print("\n[4] 2-bit per-tensor quantisation")
    q2 = make_quantise_fn("per_tensor", bits=2)
    ppl_2bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q2)
    print(f"    perplexity = {ppl_2bit:.2f}")
    assert ppl_2bit >= ppl_8bit, "2-bit should be >= 8-bit in perplexity"
    print("    ✓")

    # ── Test 5: SDS sanity check ──────────────────────────────────────────────
    print("\n[5] SDS sanity check")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q8,
                                       held_out_ids[:100])
    U_k = _compute_subspace(Z, k=32)
    sds = _sds(Z, Z_q, U_k)
    print(f"    SDS(k=32, 8-bit) = {sds:.4f}  (expect < 0.5 for 8-bit)")
    assert 0 <= sds <= 1, f"SDS={sds} outside [0,1]"
    print("    ✓")

    # ── Test 6: CKA sanity check ──────────────────────────────────────────────
    print("\n[6] CKA sanity check")
    # CKA of a tensor with itself should be 1.0
    cka_self = _cka(Z[:1000], Z[:1000])
    cka_q    = _cka(Z[:1000], Z_q[:1000])
    print(f"    CKA(Z, Z)   = {cka_self:.4f}  (expect 1.0)")
    print(f"    CKA(Z, Z_q) = {cka_q:.4f}    (expect < 1.0)")
    assert abs(cka_self - 1.0) < 1e-3, f"CKA(self)={cka_self}, expected 1.0"
    assert cka_q <= 1.0
    print("    ✓")

    print("\n✓ all smoke tests passed\n")
    return ppl   # return baseline for reference

In [ ]:
# ── 5. Quantisation functions ─────────────────────────────────────────────────
def _uniform_quantise(z: torch.Tensor, delta: torch.Tensor,
                      bits: int) -> torch.Tensor:
    """
    Apply uniform quantisation given a pre-computed step size delta.
    Formula (from assignment key formulas):
        ẑ = clip(round(z / Δ), q_min, q_max)
        z̃ = Δ * ẑ
    """
    q_min = -(2 ** (bits - 1))
    q_max =  (2 ** (bits - 1)) - 1
    z_scaled = z / delta.clamp(min=1e-8)
    z_clipped = torch.clamp(torch.round(z_scaled), q_min, q_max)
    return delta * z_clipped


def _calibrate(z: torch.Tensor, quant_type: str, bits: int) -> torch.Tensor:
    """
    Compute per-tensor or per-feature step size Δ from min/max calibration.
    Per-tensor: one Δ for the whole matrix.
    Per-feature: one Δ per feature dimension (column of z).
    """
    n_levels = 2 ** bits - 1
    if quant_type == "per_tensor":
        delta = (z.max() - z.min()) / n_levels
        return delta.expand(z.shape[-1])   # broadcast to feature dim
    elif quant_type == "per_feature":
        # z: (N, m) — compute min/max along the token dimension
        delta = (z.max(dim=0).values - z.min(dim=0).values) / n_levels
        return delta   # (m,)
    else:
        raise ValueError(f"Unknown quant_type: {quant_type}")


def make_quantise_fn(quant_type: str, bits: int,
                     calibration_z: Optional[torch.Tensor] = None
                     ) -> Callable:
    """
    Returns a quantisation function z → z_q that can be passed to the
    patching hook. Calibration uses the provided tensor if given; otherwise
    calibrates on each batch independently (less accurate but workable for
    smoke tests where we don't have a calibration set yet).
    """
    _delta = None
    if calibration_z is not None:
        _delta = _calibrate(calibration_z, quant_type, bits)

    def quantise_fn(z: torch.Tensor) -> torch.Tensor:
        nonlocal _delta
        delta = _delta if _delta is not None else _calibrate(z, quant_type, bits)
        return _uniform_quantise(z, delta.to(z.device), bits)

    return quantise_fn

In [ ]:
# ── 5b. Vector Quantisation ───────────────────────────────────────────────────
def fit_vq_codebook(calibration_z: torch.Tensor, n_codes: int) -> torch.Tensor:
    """
    Fit a VQ codebook via k-means on calibration bottleneck activations.
    Returns codebook: (n_codes, m) float32.

    VQ replaces each activation vector with its nearest codebook entry,
    exploiting inter-dimensional correlations rather than quantising each
    dimension independently. Effective bit rate is log2(n_codes) bits per
    vector — far more aggressive than per-dimension uniform quantisation.
    A 256-entry codebook gives 8 bits per m-dim vector vs 8-bit uniform
    which gives 8 bits per dimension (8m bits per vector total).
    """
    from sklearn.cluster import MiniBatchKMeans
    print(f"  Fitting VQ codebook: {n_codes} codes on {len(calibration_z):,} activations...")
    km = MiniBatchKMeans(n_clusters=n_codes, batch_size=4096,
                         n_init=3, random_state=42)
    km.fit(calibration_z.numpy())
    codebook = torch.tensor(km.cluster_centers_, dtype=torch.float32)
    print(f"  Codebook fitted: {codebook.shape}  inertia={km.inertia_:.2f}")
    return codebook


def make_vq_fn(codebook: torch.Tensor) -> Callable:
    """
    Returns a quantisation function: replaces each row of z with its
    nearest codebook entry by L2 distance.
    """
    cb = codebook.to(DEVICE)   # (n_codes, m)

    def vq_fn(z: torch.Tensor) -> torch.Tensor:
        # ||z - c||^2 = ||z||^2 + ||c||^2 - 2 z·c^T
        dists = (
            (z ** 2).sum(dim=1, keepdim=True)
            + (cb ** 2).sum(dim=1, keepdim=True).T
            - 2 * (z @ cb.T)
        )                              # (batch, n_codes)
        return cb[dists.argmin(dim=1)] # (batch, m)

    return vq_fn

In [ ]:
# ── 6. Hook infrastructure ────────────────────────────────────────────────────
class CaptureHook:
    """
    Captures layer-3 hidden states without modifying them.
    Used to collect full-precision activations for metric computation.
    """
    def __init__(self):
        self.activations: list[torch.Tensor] = []
        self._handle = None

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            lambda m, inp, out: self.activations.append(
                out[0].detach().cpu().to(torch.float16)
            )
        )
        return self

    def pop(self) -> torch.Tensor:
        out = torch.cat(self.activations, dim=0)
        self.activations.clear()
        return out

    def remove(self):
        if self._handle:
            self._handle.remove()


class PatchingHook:
    """
    Replaces layer-3 hidden states with SAE-reconstructed (optionally
    quantised) activations. The patched activations flow through layers 4-5
    and the LM head, so the final perplexity reflects the information loss
    from quantisation.

    Also stores captured bottleneck activations (before and after quantisation)
    for SDS and CKA computation.
    """
    def __init__(self, sae, normalizer, quantise_fn=None):
        self.sae         = sae
        self.normalizer  = normalizer
        self.quantise_fn = quantise_fn
        self._handle     = None
        self.z_clean: list[torch.Tensor] = []    # full-precision bottleneck
        self.z_quant: list[torch.Tensor] = []    # quantised bottleneck

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            self._fn
        )
        return self

    @torch.no_grad()
    def _fn(self, module, input, output):
        h     = output[0]                        # (batch, seq_len, d_model)
        shape = h.shape
        h_flat = h.reshape(-1, D_MODEL).float()  # (batch*seq_len, d_model)

        # normalize → encode → (optionally quantise) → decode → denormalize
        h_norm = self.normalizer(h_flat)
        z      = self.sae.encode(h_norm)

        z_q = self.quantise_fn(z) if self.quantise_fn is not None else z

        self.z_clean.append(z.cpu().half())
        self.z_quant.append(z_q.cpu().half())

        h_recon = self.normalizer.inverse(self.sae.decode(z_q))
        h_recon = h_recon.reshape(shape).to(h.dtype)

        return (h_recon,) + output[1:]

    def pop_bottlenecks(self):
        z_c = torch.cat(self.z_clean, dim=0).float()
        z_q = torch.cat(self.z_quant, dim=0).float()
        self.z_clean.clear()
        self.z_quant.clear()
        return z_c, z_q

    def remove(self):
        if self._handle:
            self._handle.remove()

In [ ]:
# ── 7. Metrics ────────────────────────────────────────────────────────────────
@torch.no_grad()
def _perplexity_clean(model, input_ids: torch.Tensor) -> float:
    """Baseline perplexity with no patching."""
    total_loss, n = 0.0, 0
    for i in range(0, len(input_ids), EVAL_BATCH):
        batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
        loss  = model(batch, labels=batch).loss
        total_loss += loss.item()
        n += 1
    return math.exp(total_loss / n)


@torch.no_grad()
def _perplexity_patched(model, sae, normalizer, input_ids: torch.Tensor,
                        quantise_fn=None) -> float:
    """Perplexity with SAE reconstruction (and optional quantisation) patched in."""
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    total_loss, n = 0.0, 0
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            loss  = model(batch, labels=batch).loss
            total_loss += loss.item()
            n += 1
    finally:
        hook.remove()
    return math.exp(total_loss / n)


@torch.no_grad()
def _collect_bottleneck_pairs(model, sae, normalizer, quantise_fn,
                               input_ids: torch.Tensor):
    """
    Collect full-precision and quantised bottleneck activations on a batch.
    Returns Z (N, m) and Z_q (N, m) in float32.
    """
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            model(batch)   # forward pass; hook captures bottlenecks
    finally:
        hook.remove()
    return hook.pop_bottlenecks()


def _compute_subspace(Z: torch.Tensor, k: int) -> torch.Tensor:
    """
    Compute top-k right singular vectors of centred full-precision activations.
    Returns U_k: (m, k) — the subspace basis.
    This is fit once on the full-precision activations and reused for all
    quantised variants, so SDS measures distortion relative to the same basis.
    """
    Z_centred = Z - Z.mean(dim=0, keepdim=True)
    # Use float32 for numerical stability in SVD
    _, _, Vt = torch.linalg.svd(Z_centred.float(), full_matrices=False)
    return Vt[:k].T   # (m, k) — right singular vectors as columns


def _sds(Z: torch.Tensor, Z_q: torch.Tensor, U_k: torch.Tensor) -> float:
    """
    Subspace Distortion Score (SDS), formula from assignment key formulas:
        SDS_k = ||(Z - Ẑ) U_k||²_F  /  ||Z U_k||²_F

    Lower is better. Measures what fraction of quantisation error falls inside
    the information-carrying top-k subspace of the full-precision activations.
    """
    diff      = (Z - Z_q).float() @ U_k.float()   # (N, k)
    numerator = diff.norm(p="fro") ** 2

    proj      = Z.float() @ U_k.float()           # (N, k)
    denom     = proj.norm(p="fro") ** 2

    return (numerator / denom.clamp(min=1e-8)).item()


def _cka(X: torch.Tensor, Y: torch.Tensor) -> float:
    """
    Linear Centered Kernel Alignment between X: (N, p) and Y: (N, q).
    Measures representation similarity; 1.0 = identical, 0.0 = orthogonal.

    We use the HSIC estimator:
        CKA = HSIC(K, L) / sqrt(HSIC(K,K) * HSIC(L,L))
        K = X X^T,  L = Y Y^T  (Gram matrices)
    Centering is applied via the H matrix (H = I - 1/n * 11^T).

    We work on a CPU subset to avoid (N×N) VRAM blowup.
    """
    X, Y = X.float().cpu(), Y.float().cpu()
    n = X.shape[0]

    K = X @ X.T   # (n, n)
    L = Y @ Y.T

    # Centre: K_c = H K H,  H = I - 1/n * 11^T
    col_mean_K = K.mean(dim=0, keepdim=True)
    row_mean_K = K.mean(dim=1, keepdim=True)
    grand_K    = K.mean()
    Kc = K - col_mean_K - row_mean_K + grand_K

    col_mean_L = L.mean(dim=0, keepdim=True)
    row_mean_L = L.mean(dim=1, keepdim=True)
    grand_L    = L.mean()
    Lc = L - col_mean_L - row_mean_L + grand_L

    hsic_kl = (Kc * Lc).sum() / ((n - 1) ** 2)
    hsic_kk = (Kc * Kc).sum() / ((n - 1) ** 2)
    hsic_ll = (Lc * Lc).sum() / ((n - 1) ** 2)

    denom = (hsic_kk * hsic_ll).sqrt().clamp(min=1e-8)
    return (hsic_kl / denom).item()


def _mse(Z: torch.Tensor, Z_q: torch.Tensor) -> float:
    return F.mse_loss(Z_q.float(), Z.float()).item()

In [ ]:
# ── 8. Experiment loop ────────────────────────────────────────────────────────
def run_experiments(model, sae, normalizer, held_out_ids, baseline_ppl):
    """
    Sweep over all (quant_type, bitwidth) combinations.
    Returns a list of result dicts for table generation.
    """
    # Collect a calibration batch for computing Δ from clean activations
    # (more accurate than calibrating per-batch)
    print("Collecting calibration activations...")
    cap = CaptureHook().register(model)
    with torch.no_grad():
        for i in range(0, min(len(held_out_ids), 200), EVAL_BATCH):
            batch = held_out_ids[i : i + EVAL_BATCH].to(DEVICE)
            model(batch)
    cap.remove()
    calib_hidden = cap.pop()                    # (n_tokens, seq_len, d_model)
    calib_flat   = calib_hidden.reshape(-1, D_MODEL).float()
    calib_norm   = normalizer(calib_flat.to(DEVICE))
    with torch.no_grad():
        calib_z = sae.encode(calib_norm).cpu()  # (N, m) — calibration bottlenecks
    print(f"  calibration set: {calib_z.shape[0]:,} activation vectors\n")

    # Fit the full-precision subspace basis (used by all SDS computations)
    print("Computing full-precision subspace bases...")
    subspaces = {k: _compute_subspace(calib_z, k) for k in SDS_K_VALUES}
    print(f"  subspaces fitted for k = {SDS_K_VALUES}\n")

    results = []

    # Also record the identity-patch (no quantisation) as a reference row
    print("Running identity patch (SAE reconstruction, no quantisation)...")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids,
                                   quantise_fn=None)
    results.append({
        "quant_type": "none", "bits": "—",
        "ppl": ppl_sae, "ppl_delta_pct": 100 * (ppl_sae / baseline_ppl - 1),
        "mse": 0.0, "cka": 1.0,
        **{f"sds_{k}": 0.0 for k in SDS_K_VALUES},
    })
    print(f"  PPL = {ppl_sae:.2f}  ({results[-1]['ppl_delta_pct']:+.1f}%)\n")

    for qtype in QUANT_TYPES:
        for bits in BITWIDTHS:
            label = f"{qtype} {bits}-bit"
            print(f"Running {label}...")

            q_fn = make_quantise_fn(qtype, bits, calibration_z=calib_z.to(DEVICE))

            # Perplexity (most expensive — full held-out set)
            ppl = _perplexity_patched(model, sae, normalizer, held_out_ids,
                                       quantise_fn=q_fn)

            # Collect bottleneck pairs for MSE / CKA / SDS
            Z, Z_q = _collect_bottleneck_pairs(
                model, sae, normalizer, q_fn,
                held_out_ids[:METRIC_TOKENS // SEQ_LEN]
            )

            mse = _mse(Z, Z_q)
            # CKA on a 1000-token subset (N×N Gram matrix is expensive)
            cka = _cka(Z[:1000], Z_q[:1000])
            sds_vals = {k: _sds(Z, Z_q, subspaces[k]) for k in SDS_K_VALUES}

            row = {
                "quant_type": qtype, "bits": bits,
                "ppl": ppl, "ppl_delta_pct": 100 * (ppl / baseline_ppl - 1),
                "mse": mse, "cka": cka,
                **{f"sds_{k}": v for k, v in sds_vals.items()},
            }
            results.append(row)

            print(
                f"  PPL={ppl:.2f} ({row['ppl_delta_pct']:+.1f}%)  "
                f"MSE={mse:.4f}  CKA={cka:.4f}  "
                f"SDS@32={sds_vals[32]:.4f}  "
                f"SDS@64={sds_vals[64]:.4f}  "
                f"SDS@128={sds_vals[128]:.4f}"
            )

    # ── Vector Quantisation ───────────────────────────────────────────────────
    for n_codes in [256, 1024]:
        label = f"VQ ({n_codes} codes)"
        print(f"Running {label}...")
        codebook = fit_vq_codebook(calib_z, n_codes=n_codes)
        q_fn = make_vq_fn(codebook)

        ppl = _perplexity_patched(model, sae, normalizer, held_out_ids,
                                   quantise_fn=q_fn)
        Z, Z_q = _collect_bottleneck_pairs(
            model, sae, normalizer, q_fn,
            held_out_ids[:METRIC_TOKENS // SEQ_LEN]
        )
        mse      = _mse(Z, Z_q)
        cka      = _cka(Z[:1000], Z_q[:1000])
        sds_vals = {k: _sds(Z, Z_q, subspaces[k]) for k in SDS_K_VALUES}

        row = {
            "quant_type": f"vq_{n_codes}",
            "bits": f"~{int(math.log2(n_codes))}b/vec",
            "ppl": ppl,
            "ppl_delta_pct": 100 * (ppl / baseline_ppl - 1),
            "mse": mse, "cka": cka,
            **{f"sds_{k}": v for k, v in sds_vals.items()},
        }
        results.append(row)
        print(
            f"  PPL={ppl:.2f} ({row['ppl_delta_pct']:+.1f}%)  "
            f"MSE={mse:.4f}  CKA={cka:.4f}  "
            f"SDS@32={sds_vals[32]:.4f}  SDS@64={sds_vals[64]:.4f}  "
            f"SDS@128={sds_vals[128]:.4f}"
        )

    return results, calib_z, subspaces

In [ ]:
# ── 9. UMAP visualisation ─────────────────────────────────────────────────────
def plot_umap(model, sae, normalizer, held_out_ids, out_dir: Path):
    """
    Fit UMAP on full-precision bottleneck activations, then project and overlay
    4-bit and 2-bit variants. Each point is one token's SAE bottleneck vector.

    Why bottleneck activations rather than hidden states?
    The SAE bottleneck Z is in a sparse, interpretable space — the UMAP plot
    there is more meaningful than in the dense 768-dim hidden space.
    """
    try:
        from umap import UMAP
    except ImportError:
        print("  umap-learn not installed — skipping UMAP (pip install umap-learn)")
        return

    n_seqs = UMAP_TOKENS // SEQ_LEN
    input_ids = held_out_ids[:n_seqs]

    print("Collecting bottleneck activations for UMAP...")
    Z_all    = {}   # dict: label → (N, m) tensor

    # Full-precision
    Z_fp, _  = _collect_bottleneck_pairs(model, sae, normalizer, None, input_ids)
    Z_all["Full precision"] = Z_fp

    for bits in [4, 2]:
        q_fn = make_quantise_fn("per_tensor", bits, calibration_z=Z_fp[:5000].to(DEVICE))
        _, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q_fn, input_ids)
        Z_all[f"{bits}-bit"] = Z_q

    print(f"  fitting UMAP on {len(Z_fp):,} tokens...")
    reducer = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    Z_fp_np = Z_all["Full precision"].numpy()
    reducer.fit(Z_fp_np)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    colors = ["#2196F3", "#FF9800", "#F44336"]

    for ax, (label, Z), color in zip(axes, Z_all.items(), colors):
        emb = reducer.transform(Z.numpy())
        ax.scatter(emb[:, 0], emb[:, 1], s=0.5, alpha=0.3, c=color)
        ax.set_title(label, fontsize=13)
        ax.set_xlabel("UMAP-1")
        ax.set_ylabel("UMAP-2")
        ax.set_aspect("equal", "datalim")

    fig.suptitle("UMAP of SAE bottleneck activations (layer 3, distilgpt2)",
                 fontsize=14)
    plt.tight_layout()
    path = out_dir / "umap_quantisation.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  UMAP saved → {path}")

In [ ]:
# ── 10. Results table ──────────────────────────────────────────────────────────
def print_results_table(results: list[dict], baseline_ppl: float):
    """Print a formatted comparative table and save as CSV."""
    header = (
        f"{'Quant type':<15} {'Bits':>4}  {'PPL':>7}  {'ΔPPL%':>7}  "
        f"{'MSE':>7}  {'CKA':>6}  "
        f"{'SDS@32':>7}  {'SDS@64':>7}  {'SDS@128':>8}"
    )
    sep = "-" * len(header)
    print(f"\n{'='*len(header)}")
    print(f"Baseline (clean) PPL = {baseline_ppl:.2f}")
    print(header)
    print(sep)
    for r in results:
        print(
            f"{r['quant_type']:<15} {str(r['bits']):>4}  "
            f"{r['ppl']:>7.2f}  {r['ppl_delta_pct']:>+7.1f}  "
            f"{r['mse']:>7.4f}  {r['cka']:>6.4f}  "
            f"{r.get('sds_32', 0):>7.4f}  "
            f"{r.get('sds_64', 0):>7.4f}  "
            f"{r.get('sds_128', 0):>8.4f}"
        )
    print(sep)

    # Save as CSV for LaTeX report
    import csv
    path = OUT_DIR / "quantisation_results.csv"
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    print(f"\n  results saved → {path}")

In [ ]:
# ── Smoke test cell (run alone before the full orchestration) ──────────────
model, tokenizer = load_frozen_model()
normalizer       = ActivationNormalizer.load(NOTEBOOK01_DIR / "normalizer.pt")
sae              = load_sae(NOTEBOOK02_DIR / "sae_m512_ckpt_final.pt")

# 200 sequences is enough for every assertion in run_smoke_tests()
held_out_smoke = collect_held_out(model, tokenizer, n_seqs=200)

baseline_ppl = run_smoke_tests(model, tokenizer, sae, normalizer, held_out_smoke)

In [ ]:
# ── Full orchestration (run after smoke tests pass) ────────────────────────
held_out_ids = collect_held_out(model, tokenizer, n_seqs=HELD_OUT_SEQS)

results, calib_z, subspaces = run_experiments(
    model, sae, normalizer, held_out_ids, baseline_ppl
)

print_results_table(results, baseline_ppl)

plot_umap(model, sae, normalizer, held_out_ids, OUT_DIR)